# Dental Scheduler — Logic Tests

Testing the core scheduling logic: procedure lookups, overlap detection, booking, and conflict handling.

In [ ]:
import sys
sys.path.insert(0, "/home/arc314/Desktop/Dev/Dental_Scheduler")

from database import Base, engine, SessionLocal
from models import Doctor, Room, Patient, Appointment, doctor_procedures, room_procedures
import scheduler

# Fresh in-memory state — recreate all tables for a clean test run
Base.metadata.drop_all(bind=engine)
Base.metadata.create_all(bind=engine)
print("Tables created.")

## 1. Seed test data

In [ ]:
from sqlalchemy import select

db = SessionLocal()

# Doctors
dr_palash = Doctor(name="Dr. Palash Mehta")
dr_varun  = Doctor(name="Dr. Varun Kumar")
db.add_all([dr_palash, dr_varun])
db.flush()

db.execute(doctor_procedures.insert(), [
    {"doctor_id": dr_palash.id, "procedure": "cleaning"},
    {"doctor_id": dr_palash.id, "procedure": "filling"},
    {"doctor_id": dr_varun.id,  "procedure": "extraction"},
])

# Rooms
room1 = Room(name="Room 1")
room2 = Room(name="Room 2")
db.add_all([room1, room2])
db.flush()

db.execute(room_procedures.insert(), [
    {"room_id": room1.id, "procedure": "cleaning"},
    {"room_id": room1.id, "procedure": "filling"},
    {"room_id": room2.id, "procedure": "extraction"},
])

# Patient
john = scheduler.get_or_create_patient(db, "John Doe", phone="555-1001")

db.commit()
print(f"Doctors: {dr_palash.id}, {dr_varun.id}")
print(f"Rooms:   {room1.id}, {room2.id}")
print(f"Patient: {john.id} — {john.name}")

## 2. Procedure lookup — who can do what

In [ ]:
docs = scheduler.get_doctors_for_procedure(db, "cleaning")
print("Doctors for cleaning:", [d.name for d in docs])

docs = scheduler.get_doctors_for_procedure(db, "extraction")
print("Doctors for extraction:", [d.name for d in docs])

rooms = scheduler.get_rooms_for_procedure(db, "cleaning")
print("Rooms for cleaning:", [r.name for r in rooms])

# Should return empty — nobody is seeded for xray
docs = scheduler.get_doctors_for_procedure(db, "xray")
print("Doctors for xray:", [d.name for d in docs])

## 3. Book a valid appointment

In [ ]:
from datetime import datetime

slot = scheduler.find_available_slot(
    db, "cleaning",
    start=datetime(2026, 6, 1, 10, 0),
    end=datetime(2026, 6, 1, 11, 0),
)
print("Available slot:", slot)

if slot:
    doctor, room = slot
    appt = scheduler.book_appointment(
        db,
        patient_id=john.id,
        doctor_id=doctor.id,
        room_id=room.id,
        procedure="cleaning",
        start=datetime(2026, 6, 1, 10, 0),
        end=datetime(2026, 6, 1, 11, 0),
    )
    print(f"Booked appointment #{appt.id}: {doctor.name} in {room.name} at {appt.start_time}")

## 4. Overlap detection — attempt to double-book the same slot

In [ ]:
# Try to book an overlapping slot — 10:30am to 11:30am, same doctor
try:
    scheduler.book_appointment(
        db,
        patient_id=john.id,
        doctor_id=dr_palash.id,
        room_id=room1.id,
        procedure="cleaning",
        start=datetime(2026, 6, 1, 10, 30),
        end=datetime(2026, 6, 1, 11, 30),
    )
    print("ERROR: should have raised a conflict!")
except ValueError as e:
    print(f"Conflict correctly caught: {e}")

# Adjacent slot (11am–12pm) should succeed — no overlap
appt2 = scheduler.book_appointment(
    db,
    patient_id=john.id,
    doctor_id=dr_palash.id,
    room_id=room1.id,
    procedure="filling",
    start=datetime(2026, 6, 1, 11, 0),
    end=datetime(2026, 6, 1, 12, 0),
)
print(f"Adjacent slot booked OK: #{appt2.id} at {appt2.start_time}")

## 5. Retrieve patient appointments

In [ ]:
appts = scheduler.get_patient_appointments(db, john.id)
print(f"Upcoming appointments for {john.name}:")
for a in appts:
    print(f"  #{a.id} — {a.procedure} with {a.doctor.name} in {a.room.name} @ {a.start_time} → {a.end_time}")

db.close()